# Chicago Crime · Building a Consistent Public Data Snapshot
**Hesub Chaka | Python, SQL, data validation**

A useful analysis starts with a question and a trustworthy denominator. This project asks how Chicago’s reported crime patterns and recorded arrest shares vary over time, across categories, and across police districts.

This public project independently retrieves compact aggregate tables from the City of Chicago. All numerical results come from the public snapshot included in this repository.

## Source and unit of analysis
[Chicago Data Portal: Crimes—2001 to Present](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2/about_data)

**Window:** January 1, 2001 through November 23, 2025 inclusive. Charts use complete years through 2024. The public source contains incident records, with victim-level records for murders; records are not necessarily distinct cases. Arrest is a case attribute and may be revised after the incident.

Socrata performs SQL-like aggregation on the server. Four CSVs contain annual, monthly, district/year and category/year summaries. No individual records, addresses, or credentials are committed. Raw public district strings are preserved in the downloaded CSV; the local preparation step converts numeric variants such as `016` and `16` to a common code and sums their counts. Missing districts remain in reconciliation totals.

The manifest records every query, retrieval timestamp, returned row count and file checksum. The saved CSVs are the reproducible input; refreshing the live source can change older years.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'scripts').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
import pandas as pd
from analyze import load, SQL_ANNUAL, SQL_TRENDS
frames, db = load()


In [2]:
manifest = json.loads((ROOT / 'data/manifest.json').read_text())
print('Retrieved:', manifest['completed_utc'])
print('Date filter:', manifest['date_filter'])
print(json.dumps(json.loads((ROOT / 'data/source_check.json').read_text()), indent=2))

Retrieved: 2026-10-22T20:01:28.950622+00:00
Date filter: date >= '2001-01-01T00:00:00' AND date < '2025-11-24T00:00:00'
{
  "records": "8459600",
  "distinct_cases": "8458992",
  "earliest": "2001-01-01T00:00:00.000",
  "latest": "2025-11-23T23:58:00.000"
}


## Extraction logic
These queries are the actual requests used for the saved data. `case(arrest = true, 1, true, 0)` uses Socrata syntax, which differs from PostgreSQL `CASE WHEN` syntax.

For an intentional refresh, run `python scripts/fetch_data.py` from the project root. Refreshing overwrites the saved aggregate snapshot; retain the old manifest if comparing versions.

In [3]:
from fetch_data import QUERIES
for name, query in QUERIES.items():
    print(name, json.dumps(query), sep=': ')

annual: {"$select": "year, count(*) AS incidents, sum(case(arrest = true, 1, true, 0)) AS arrests", "$group": "year", "$order": "year"}
monthly: {"$select": "year, date_extract_m(date) AS month, count(*) AS incidents, sum(case(arrest = true, 1, true, 0)) AS arrests", "$group": "year, month", "$order": "year, month"}
district_annual: {"$select": "year, district, count(*) AS incidents, sum(case(arrest = true, 1, true, 0)) AS arrests", "$group": "year, district", "$order": "year, district"}
category_annual: {"$select": "year, primary_type, count(*) AS incidents, sum(case(arrest = true, 1, true, 0)) AS arrests", "$group": "year, primary_type", "$order": "year, primary_type"}
source_check: {"$select": "count(*) AS records, count(distinct case_number) AS distinct_cases, min(date) AS earliest, max(date) AS latest"}


## Validate before interpreting
The pipeline checks unique aggregate keys, nonnegative counts, arrest counts no greater than total counts, full month coverage, and exact annual reconciliation across all four rollups. The public record total is checked independently against the API count query. These checks validate consistency of the saved public summaries.

The live API is not an atomic snapshot across multiple requests. Reconciliation detects changed totals between requests; source values could still be updated without changing totals.

In [4]:
checks = []
base = frames['annual'].set_index('year')[['incidents','arrests']]
for name in ['monthly','district_annual','category_annual']:
    totals = frames[name].groupby('year')[['incidents','arrests']].sum()
    checks.append({'table':name,'annual_totals_match':totals.equals(base), 'rows_after_preparation':len(frames[name])})
pd.DataFrame(checks)

,table,annual_totals_match,rows_after_preparation
0,monthly,True,299
1,district_annual,True,607
2,category_annual,True,775


## A dated public snapshot
This public extraction covers **8,459,600 records** and **8,458,992 distinct case numbers**. Source records can be revised or reported late; preserve retrieval timestamps when comparing versions. Both the date scope and distinction between cases and records are retained explicitly.

Category analysis uses the public source’s `primary_type` labels. No unverified FBI/NIBRS crosswalk is reconstructed. Next: use local SQL to turn the validated public summaries into interpretable comparisons.